# Task 3 - Forecast Future Market Trends
**Objective:** Use trained models to forecast Tesla's future stock prices and translate results into actionable business insights.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sys, os
sys.path.append('../src')

import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

from data_loader import load_cleaned
from arima_model import find_best_order, fit_sarimax, forecast as arima_test_forecast
from lstm_model  import prepare_sequences, build_model, train_model, generate_forecast
from evaluate    import compute_metrics, build_comparison_table, plot_all_forecasts, plot_metrics_bar, print_summary
from forecast    import (
    generate_future_dates,
    arima_future_forecast,
    lstm_future_forecast,
    plot_arima_future,
    plot_lstm_future,
    plot_ci_width_over_time,
    trend_analysis
)

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 2. Load Data and Prepare Train / Test Split

In [ ]:
tsla = load_cleaned('TSLA')[['Adj Close']].rename(columns={'Adj Close': 'Close'})
print(f"TSLA data: {tsla.shape}  |  {tsla.index.min().date()} to {tsla.index.max().date()}")

SPLIT_DATE = '2025-01-01'
train = tsla[tsla.index < SPLIT_DATE]['Close']
test  = tsla[tsla.index >= SPLIT_DATE]['Close']

print(f"Train: {len(train)} rows  |  Test: {len(test)} rows")

## 3. Retrain Models on Full Training Data

### 3.1 ARIMA / SARIMA

In [ ]:
# Find best ARIMA order
auto = find_best_order(train, seasonal=False)
best_order = auto.order
print(f"Best ARIMA order: {best_order}")

# Fit on training data
arima_fitted = fit_sarimax(train, order=best_order)

# Test period forecast (for evaluation)
arima_test_fc = arima_test_forecast(arima_fitted, steps=len(test), index=test.index)
print("ARIMA fitted and test forecast generated.")

### 3.2 SARIMA (Seasonal)

In [ ]:
sarima_auto = find_best_order(train, seasonal=True, m=12)
sarima_order          = sarima_auto.order
sarima_seasonal_order = sarima_auto.seasonal_order
print(f"Best SARIMA: {sarima_order} x {sarima_seasonal_order}")

sarima_fitted = fit_sarimax(train, order=sarima_order, seasonal_order=sarima_seasonal_order)
sarima_test_fc = arima_test_forecast(sarima_fitted, steps=len(test), index=test.index)
print("SARIMA fitted and test forecast generated.")

### 3.3 LSTM

In [ ]:
WINDOW = 60
X_train, y_train, X_test, y_test, scaler = prepare_sequences(train, test, window=WINDOW)

lstm_model = build_model(window=WINDOW)
history    = train_model(lstm_model, X_train, y_train)

lstm_test_preds = generate_forecast(lstm_model, X_test, scaler)
print(f"LSTM trained. Test predictions shape: {lstm_test_preds.shape}")

## 4. Model Evaluation on Test Period

In [ ]:
actual = test.values

results = [
    compute_metrics(actual, arima_test_fc['forecast'].values,  f'ARIMA{best_order}'),
    compute_metrics(actual, sarima_test_fc['forecast'].values, f'SARIMA{sarima_order}x{sarima_seasonal_order}'),
    compute_metrics(actual, lstm_test_preds,                    'LSTM (2-layer)')
]

results_df = build_comparison_table(results)
print_summary(results_df)

best_model_name = results_df['RMSE'].idxmin()
print(f"\nBest model selected for future forecasting: {best_model_name}")

In [ ]:
plot_all_forecasts(
    test.index, actual,
    forecasts={
        f'ARIMA{best_order}':  arima_test_fc['forecast'].values,
        'SARIMA':              sarima_test_fc['forecast'].values,
        'LSTM':                lstm_test_preds
    }
)
plot_metrics_bar(results_df)

## 5. Generate Future Forecasts (6–12 Months)

### 5.1 ARIMA Future Forecast

In [ ]:
# Refit ARIMA on ALL available data (train + test) for future forecasting
from statsmodels.tsa.statespace.sarimax import SARIMAX

full_series = tsla['Close']

arima_full = SARIMAX(
    full_series,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

arima_future_df = arima_future_forecast(arima_full, last_date=full_series.index[-1], n_months=12)

print(f"ARIMA future forecast: {len(arima_future_df)} trading days")
print(f"Forecast period: {arima_future_df.index[0].date()} to {arima_future_df.index[-1].date()}")
arima_future_df.head()

### 5.2 LSTM Future Forecast (Iterative Multi-Step)

In [ ]:
# Refit LSTM on ALL available data
from sklearn.preprocessing import MinMaxScaler

full_scaler = MinMaxScaler(feature_range=(0, 1))
full_scaled = full_scaler.fit_transform(full_series.values.reshape(-1, 1))

# Retrain on full data
def make_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i - window:i, 0])
        y.append(data[i, 0])
    return np.array(X).reshape(-1, window, 1), np.array(y)

X_full, y_full = make_sequences(full_scaled, WINDOW)
lstm_full = build_model(window=WINDOW)
train_model(lstm_full, X_full, y_full)

# Last window of scaled prices as seed
last_window = full_scaled[-WINDOW:, 0].tolist()

lstm_future_prices, lstm_lower, lstm_upper = lstm_future_forecast(
    lstm_full, last_window, full_scaler, n_months=12
)

future_dates = generate_future_dates(full_series.index[-1], len(lstm_future_prices))

print(f"LSTM future forecast: {len(lstm_future_prices)} trading days")
print(f"Forecast period: {future_dates[0].date()} to {future_dates[-1].date()}")

## 6. Visualize Forecasts with Confidence Intervals

### 6.1 ARIMA Forecast Plot

In [ ]:
plot_arima_future(
    history       = full_series,
    test_actual   = test,
    test_forecast_df = arima_test_fc,
    future_df     = arima_future_df,
    title         = f'ARIMA{best_order} — 12-Month Future Forecast with 95% CI'
)

### 6.2 LSTM Forecast Plot

In [ ]:
plot_lstm_future(
    history        = full_series,
    test_actual    = test,
    test_forecast  = lstm_test_preds,
    future_dates   = future_dates,
    future_forecast= lstm_future_prices,
    lower_ci       = lstm_lower,
    upper_ci       = lstm_upper
)

### 6.3 Confidence Interval Width Over Time

In [ ]:
plot_ci_width_over_time(
    future_df    = arima_future_df,
    future_dates = future_dates,
    lower_ci     = lstm_lower,
    upper_ci     = lstm_upper
)

## 7. Trend Analysis

In [ ]:
trend_analysis(arima_future_df, label=f'ARIMA{best_order} — 12-Month Future Forecast')
trend_analysis(lstm_future_prices, label='LSTM — 12-Month Future Forecast')

In [ ]:
# Monthly breakdown of ARIMA forecast
arima_monthly = arima_future_df['forecast'].resample('ME').last()
lstm_monthly  = pd.Series(lstm_future_prices, index=future_dates).resample('ME').last()

monthly_df = pd.DataFrame({
    'ARIMA Forecast': arima_monthly,
    'LSTM Forecast':  lstm_monthly
})
print("\nMonthly Forecast Summary:")
print(monthly_df.to_string())

### Trend Analysis Summary

**Short-term (1–3 months):** Both ARIMA and LSTM forecasts show relatively narrow confidence intervals in the near term, reflecting higher model certainty. ARIMA tends to project a mean-reverting path anchored to recent price levels, while LSTM captures momentum from the last 60-day window and may project continuation of the prevailing trend.

**Long-term (4–12 months):** Confidence intervals widen substantially as the forecast horizon extends. ARIMA's CI expands linearly due to the accumulation of one-step-ahead errors, while LSTM's Monte Carlo Dropout intervals widen non-linearly as iterative prediction errors compound. By month 12, the CI width may span hundreds of dollars for TSLA, reflecting the fundamental difficulty of long-horizon price forecasting for a high-volatility asset. This widening is not a model failure — it is an honest representation of forecast uncertainty and should be communicated clearly to clients.

## 8. Market Opportunities and Risks Assessment

In [ ]:
# Compute key statistics for the opportunity/risk assessment
arima_vals = arima_future_df['forecast'].values
lstm_vals  = lstm_future_prices

arima_pct = (arima_vals[-1] - arima_vals[0]) / arima_vals[0] * 100
lstm_pct  = (lstm_vals[-1]  - lstm_vals[0])  / lstm_vals[0]  * 100

arima_ci_start = (arima_future_df['upper_ci'].iloc[0]  - arima_future_df['lower_ci'].iloc[0])
arima_ci_end   = (arima_future_df['upper_ci'].iloc[-1] - arima_future_df['lower_ci'].iloc[-1])
lstm_ci_start  = lstm_upper[0]  - lstm_lower[0]
lstm_ci_end    = lstm_upper[-1] - lstm_lower[-1]

print("="*60)
print("  MARKET OPPORTUNITIES AND RISKS ASSESSMENT")
print("="*60)
print(f"\n  ARIMA 12-month projected change : {arima_pct:+.2f}%")
print(f"  LSTM  12-month projected change : {lstm_pct:+.2f}%")
print(f"\n  ARIMA CI width — Month 1 : ${arima_ci_start:.2f}")
print(f"  ARIMA CI width — Month 12: ${arima_ci_end:.2f}")
print(f"  LSTM  CI width — Month 1 : ${lstm_ci_start:.2f}")
print(f"  LSTM  CI width — Month 12: ${lstm_ci_end:.2f}")

### Opportunities

1. **Upward momentum entry point** — If both ARIMA and LSTM project a positive 12-month return, the near-term (1–3 month) window with narrow CIs represents the highest-confidence entry opportunity for long positions.
2. **Volatility premium** — TSLA's high historical volatility (~50% annualized) creates opportunities for options strategies (e.g., covered calls, straddles) that benefit from elevated implied volatility.
3. **Trend continuation** — If the LSTM forecast captures upward momentum from the last 60-day window, a momentum-based allocation increase in TSLA within the portfolio may be warranted for risk-tolerant clients.
4. **Portfolio rebalancing signal** — A forecasted price increase in TSLA relative to BND and SPY suggests a potential tactical overweight in TSLA for growth-oriented portfolios.

### Risks

1. **Widening confidence intervals** — CI width grows significantly beyond month 3, meaning long-horizon forecasts carry substantial uncertainty. Decisions based on 9–12 month forecasts should be treated as directional guidance only, not precise targets.
2. **Model limitations** — ARIMA assumes linear relationships and cannot capture sudden regime changes (e.g., earnings shocks, regulatory events). LSTM's iterative forecasting compounds errors over time, potentially drifting from true prices.
3. **High volatility risk** — TSLA's ~50% annualized volatility means even a "correct" directional forecast can be accompanied by severe drawdowns. The VaR (95%) of ~-3.5% daily implies significant short-term loss potential.
4. **Efficient Market Hypothesis caveat** — Per the EMH, these forecasts should not be used as standalone trading signals. They are most reliable as inputs into a broader risk management and portfolio construction framework.

### Forecast Reliability Assessment

| Horizon | Reliability | Recommended Use |
|---------|------------|------------------|
| 1–4 weeks | Moderate–High | Short-term tactical allocation |
| 1–3 months | Moderate | Portfolio rebalancing triggers |
| 3–6 months | Low–Moderate | Directional bias only |
| 6–12 months | Low | Scenario planning, stress testing |

## 9. Save All Forecast Data

In [ ]:
# Save ARIMA future forecast
arima_future_df.to_csv(f'{PROCESSED_DIR}/arima_future_forecast.csv')

# Save LSTM future forecast
lstm_future_out = pd.DataFrame({
    'forecast':  lstm_future_prices,
    'lower_ci':  lstm_lower,
    'upper_ci':  lstm_upper
}, index=future_dates)
lstm_future_out.to_csv(f'{PROCESSED_DIR}/lstm_future_forecast.csv')

print("Forecast data saved to data/processed/")
print(f"  arima_future_forecast.csv : {len(arima_future_df)} rows")
print(f"  lstm_future_forecast.csv  : {len(lstm_future_out)} rows")